# Kronos Price Forecast

Run on Colab's free **T4 GPU**: Runtime → Change runtime type → **T4 GPU**.

**Secrets** (🔑 left sidebar): `TURSO_DATABASE_URL`, `TURSO_AUTH_TOKEN` (same as `.env.local`), and `GH_TOKEN` (a fine‑grained GitHub PAT with read access to the private `trades` repo).

**To run:** edit the **CONFIG** cell, then Runtime → **Run all**. The setup cell wipes & re‑clones the repo every run and the run cell reloads the module, so you always use the latest pushed code (no manual restart needed). The notebook pulls fresh daily bars from Yahoo Finance by default, writes the forecast batch to Turso, then you can open `/report/<TICKER>` in the app and hit **Remake Report**.

In [ ]:
# Always pull the latest pushed code (fresh clone) + the Kronos model repo.
from google.colab import userdata
gh = userdata.get('GH_TOKEN')
!rm -rf trades && git clone https://{gh}@github.com/tvay11/trades.git
!git clone https://github.com/shiyu-coder/Kronos 2>/dev/null || true
%pip install -q -r trades/forecast/requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ['TURSO_DATABASE_URL'] = userdata.get('TURSO_DATABASE_URL')
os.environ['TURSO_AUTH_TOKEN'] = userdata.get('TURSO_AUTH_TOKEN')
os.environ['KRONOS_REPO'] = '/content/Kronos'

In [ ]:
# ===== CONFIG — edit these =====
TICKERS = ["AAPL"]      # one or many, e.g. ["AAPL", "NVDA", "MSFT"]
HORIZON = 60            # trading days to forecast
SAMPLES = 100           # sampled paths (more = steadier band, NOT tighter)
TEMPERATURE = 0.6       # lower = tighter band, higher = wider (try 0.4–1.0)
DATA_SOURCE = "yahoo"  # fresh Yahoo Finance daily bars; use "cache" only for TickerPriceCache fallback

In [ ]:
import sys, importlib
sys.path.insert(0, '/content/trades/forecast')
importlib.invalidate_caches()       # clear Python's import cache to find newly cloned files
import kronos_forecast
importlib.reload(kronos_forecast)   # pick up the freshly cloned code without a restart
for t in TICKERS:
    kronos_forecast.run(t, horizon=HORIZON, samples=SAMPLES, temperature=TEMPERATURE, source=DATA_SOURCE)